# 02 - Traditional baseline: SLIC superpixels + Random Forest (Part A)

**How it works:**
1. SLIC groups each image into ~150 *superpixels* (small blobs of similar colour).
2. Each superpixel is described by simple numbers: average colour (RGB and Lab), colour spread, and position.
3. A Random Forest learns: features of a superpixel -> its class (the majority class of its true pixels).
4. To label a new image: run SLIC, classify every superpixel, paint it with its predicted class.

No neural network

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import sys, os
import numpy as np
sys.path.insert(0, '/content/drive/MyDrive/coco_seg/project/code')
import config as C
import data, engine, losses, models, agent_search, report_utils
C.ensure_dirs()
print('GPU:', __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'NONE - Runtime > Change runtime type > T4 GPU')
# Baseline knobs (subset sizes keep it fast; documented in the report)
BASELINE_TRAIN_IMAGES = 300
BASELINE_VAL_IMAGES   = 150
N_SEGMENTS            = 150
N_TREES               = 100
print('baseline settings:', BASELINE_TRAIN_IMAGES, 'train imgs,', BASELINE_VAL_IMAGES, 'val imgs')

GPU: NONE - Runtime > Change runtime type > T4 GPU
baseline settings: 300 train imgs, 150 val imgs


### Feature extraction

In [5]:
import numpy as np
from PIL import Image
from skimage.segmentation import slic
from skimage.color import rgb2lab

def superpixels_and_features(pil_img, n_segments=N_SEGMENTS):
    """Returns (segments array, features per segment)."""
    img = np.asarray(pil_img.convert('RGB').resize((256, 256), Image.BILINEAR)).astype(np.float32) / 255.0
    seg = slic(img, n_segments=n_segments, compactness=10, start_label=0, channel_axis=-1)
    lab = rgb2lab(img)
    feats = []
    for s in range(seg.max() + 1):
        m = seg == s
        ys, xs = np.nonzero(m)
        rgb_mean = img[m].mean(axis=0)
        rgb_std = img[m].std(axis=0)
        lab_mean = lab[m].mean(axis=0)
        xy = np.array([xs.mean() / 256.0, ys.mean() / 256.0])
        feats.append(np.concatenate([rgb_mean, rgb_std, lab_mean, xy]))
    return seg, np.stack(feats)

def majority_labels(seg, mask, n):
    out = np.zeros(n, dtype=np.int64)
    for s in range(n):
        vals = mask[seg == s]
        out[s] = np.bincount(vals, minlength=C.NUM_CLASSES).argmax()
    return out

print('helpers defined')

helpers defined


### Train the Random Forest 

In [6]:
import os, joblib
from sklearn.ensemble import RandomForestClassifier

rf_path = os.path.join(C.CKPT_DIR, 'baseline_rf.joblib')
frames = data.load_frames()
if os.path.exists(rf_path):
    rf = joblib.load(rf_path)
    print('[resume] loaded the trained forest from Drive')
else:
    X, Y = [], []
    subset = frames['train'].sample(BASELINE_TRAIN_IMAGES, random_state=42)
    for i, f in enumerate(subset.itertuples()):
        seg, feats = superpixels_and_features(Image.open(f.image))
        mask = np.asarray(Image.open(f.mask).resize((256, 256), Image.NEAREST))
        X.append(feats)
        Y.append(majority_labels(seg, mask, len(feats)))
        if (i + 1) % 50 == 0:
            print(f'{i + 1}/{len(subset)} images')
    X, Y = np.concatenate(X), np.concatenate(Y)
    print('training set:', X.shape, 'superpixels')
    rf = RandomForestClassifier(n_estimators=N_TREES, n_jobs=-1, random_state=42,
                                class_weight='balanced_subsample')
    rf.fit(X, Y)
    joblib.dump(rf, rf_path)
    print('forest trained and saved to Drive')

masks index: 10000 rows | splits: {'train': 8000, 'val': 1000, 'test': 1000} | missing files: 0
                                                                            image                                                          mask  split
0  /content/drive/MyDrive/coco_seg/fiftyone/coco-2017/train/data/000000307967.jpg  /content/drive/MyDrive/coco_seg/masks/train/000000307967.png  train
1  /content/drive/MyDrive/coco_seg/fiftyone/coco-2017/train/data/000000154345.jpg  /content/drive/MyDrive/coco_seg/masks/train/000000154345.png  train
50/300 images
100/300 images
150/300 images
200/300 images
250/300 images
300/300 images
training set: (31787, 11) superpixels
forest trained and saved to Drive


### Evaluate on validation images

In [7]:
import metrics as M
cmsum = np.zeros((C.NUM_CLASSES, C.NUM_CLASSES), dtype=np.int64)
val_subset = frames['val'].sample(BASELINE_VAL_IMAGES, random_state=42)
examples = []
for i, f in enumerate(val_subset.itertuples()):
    img = Image.open(f.image)
    seg, feats = superpixels_and_features(img)
    pred_sp = rf.predict(feats)
    pred = pred_sp[seg]
    mask = np.asarray(Image.open(f.mask).resize((256, 256), Image.NEAREST))
    cmsum += M.confusion_matrix(pred, mask, C.NUM_CLASSES)
    if i < 3:
        examples.append((np.asarray(img.resize((256, 256))).astype(np.float32).transpose(2, 0, 1) / 255.0, mask, pred))
    if (i + 1) % 50 == 0:
        print(f'{i + 1}/{len(val_subset)} images')

res = M.metrics_from_confusion(cmsum)
res['model'] = 'SLIC superpixels + Random Forest (traditional baseline)'
res['settings'] = {'train_images': BASELINE_TRAIN_IMAGES, 'val_images': BASELINE_VAL_IMAGES,
                   'n_segments': N_SEGMENTS, 'n_trees': N_TREES}
report_utils.save_result('baseline_slic_rf', res)
print('baseline mIoU:', round(res['miou'], 4), '| pixel accuracy:', round(res['pixel_accuracy'], 4))

50/150 images
100/150 images
150/150 images
baseline mIoU: 0.0667 | pixel accuracy: 0.8201


### Save example predictions for the report

In [8]:
import os
from data_viz import overlay
for i, (img, gt, pred) in enumerate(examples):
    # img is 0..1 CHW; overlay expects ImageNet-normalised input, so re-normalise
    x = (img - np.array(C.IMAGENET_MEAN).reshape(3,1,1)) / np.array(C.IMAGENET_STD).reshape(3,1,1)
    overlay(x, gt, pred, C.CLASS_NAMES,
            os.path.join(C.FIGURES_DIR, f'baseline_example_{i}.png'),
            title=f'SLIC + Random Forest, example {i}')
print('example figures saved')

example figures saved
